---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegem două axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 11


Root project: c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[turcescu111] NU, TURUL DOI DL.GEORGESCU ESTE ADEVARATUL PRESEDINTE, DRAGILOR EL ESTE...L-A AL
[StareaNatiei] Foarte slab, filozofie de balta, si cu ochelari mari de cal ai omului care a reu
[RecorderRomania] Ce glume, ce manipulări. Înregistrări cu 41 milioane vizualizări a patrioților a


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [6]:
# modifica dupa preferinte

AXA_1 = "elite_blame"
AXA_2 = "national_identity"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [7]:
AXA_1_DEFINITION = """
elite_blame măsoară dacă textul dă vina pe elite (politice, economice, culturale) pentru problemele societății.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
national_identity măsoară dacă textul exprimă o identitate națională sau un sentiment de apartenență la o nație.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [8]:
MINI_PROMPT = f"""
Ești un expert în analiză discursivă și adnotare de text. Ai următoarele instrucțiuni clare pentru a adnota un comentariu politic de pe YouTube, folosind două axe tematice relevante pentru polarizarea discursului politic.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un expert în analiză discursivă și adnotare de text. Ai următoarele instrucțiuni clare pentru a adnota un comentariu politic de pe YouTube, folosind două axe tematice relevante pentru polarizarea discursului politic.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. elite_blame
2. national_identity
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
elite_blame = 0 / 1 / 2
national_identity = 0 / 1 / 2
DEFINIȚII:

elite_blame măsoară dacă textul dă vina pe elite (politice, economice, culturale) pentru problemele societății.
0 = absent
1 = prezent
2 = dominant


national_identity măsoară dacă textul exprimă o identitate națională sau un sentiment de apartenență la o nație.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [9]:
TESTS = corpus.sample(5, random_state=40)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
71,yt_DrY0m2OjSdQ_UgwPq1U2oRWVt2sD5Ch4AaABAg,georgesimionoficial,#georgesimion #unitate #democratie #impreuna #...,"Domnule presedinte, as vrea sa va reamintesc c..."
142,yt_97qXZe3_J_k_Ugz_-EFMa2xlR3TvdAh4AaABAg,turcescu111,Georgescu le-a dat la operație!,"Pentru ,,viata de cacat "" care va sa vie trebu..."
139,yt_Sj4fQKlMOro_UgyUQcT-gXyrgTWCVO14AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Vă rog să aveți îndurare. Nu vreau să își piar...
102,yt_XRhxwXmeOJE_Ugxocy4v8cZiZaTZzAx4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - 1 Decembrie 2025 ( 01.12.202...,"Il apreciez foarte mult pe CG, dar cu disensiu..."
228,yt_Vekhmz5OPCc_UgxjT7UCR2f7Unta4eJ4AaABAg,NicusorDanRO,🟢 LIVE Declarații de presă susținute la Palatu...,Ma bucur mult ca avem Coruptia in Strategie. P...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [10]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [11]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [12]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Domnule presedinte, as vrea sa va reamintesc ca Maia Sandu este deja presedinte in Moldova si in plus, pentru stiinta dvs rasuflata si inexistenta a votantilor dvs, este chiar la al doilea mandat. Asadar, 100.000 de voturi pentru ce si cand sa nu castige Maia Sandu? Domnule presesinte, mai aveti o ultima besina puturoasa de dat si nu va lasati pana nu o scapati. Toata lumea din Moldova te vede ca pe un cartof expirat plin de viermi, nu te mai asculta nimeni. Du-te si fa ceva in viata, sunt sigura ca nici primul ajutor nu stii sa il dai.

OUTPUT MODEL:
```json
{
  "target": "Klaus Iohannis",
  "stance": "anti",
  "tone": "acuzator",
  "elite_blame": 2,
  "national_identity": 1
}
```
COMENTARIU:
Pentru ,,viata de cacat " care va sa vie trebuie multumit prietenilor de peste Atlantic. Nimic bun nu o sa se intample. O sa fie din ce in ce mai rau.

OUTPUT MODEL:
```json
{
  "target": "SUA",
  "stance": "anti",
  "tone": "acuzator",
  "elite_blame": 1,
  "national_identity": 0
}
`

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?


Ca axe pentru model am ales "elite_blame" și "national_identity" pentru că mi s-au părut relevante pentru polarizarea discursului politic, mai ales în contextul actual. "elite_blame" măsoară dacă textul dă vina pe elite (politice, economice, culturale) pentru problemele societății, iar "national_identity" măsoară dacă textul exprimă o identitate națională sau un sentiment de apartenență la o nație.

Modelul a returnat în general JSON corect și coerent structural, respectând câmpurile cerute și valorile definite, însă tot avea probleme cu subtilitățile limbajului și cu interpretarea acestuia. Ca atare, cea mai mare problemă a fost distincția clară dintre prezent și dominant (oarecum subiectiv).

Poate o delimitare în mai multe niveluri ar ajuta modelul să facă o distincție mai corectă / justificabilă pentru mine.